In [29]:
import pandas as pd
import numpy as np
import os
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler

# Load feature engineered data
train = pd.read_csv('../data/processed/train_featured.csv')
test = pd.read_csv('../data/processed/test_featured.csv')

# Separate target variable
y_train = train['SalePrice']
train = train.drop(columns=['SalePrice'])

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("y_train shape:", y_train.shape)
print("\nExterQual sample:", train['ExterQual'].value_counts().head())

Train shape: (1458, 88)
Test shape: (1459, 88)
y_train shape: (1458,)

ExterQual sample: ExterQual
TA    906
Gd    488
Ex     50
Fa     14
Name: count, dtype: int64


In [31]:
# Quality columns with natural order
order_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 
              'HeatingQC', 'KitchenQual', 'FireplaceQu',
              'GarageQual', 'GarageCond', 'PoolQC']

quality_map = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}

for col in order_cols:
    if train[col].dtype == 'object':
        train[col] = train[col].map(quality_map)
        test[col] = test[col].map(quality_map)
    # Fill any remaining NaN with 0
    train[col] = train[col].fillna(0)
    test[col] = test[col].fillna(0)

print("Label encoding done!")
print(train[order_cols].isnull().sum())

Label encoding done!
ExterQual      0
ExterCond      0
BsmtQual       0
BsmtCond       0
HeatingQC      0
KitchenQual    0
FireplaceQu    0
GarageQual     0
GarageCond     0
PoolQC         0
dtype: int64


In [33]:
# Get remaining categorical columns
remaining_cat = train.select_dtypes(include='object').columns
print("Remaining categorical columns to encode:", len(remaining_cat))

# Combine train and test before encoding
train_size = len(train)
combined = pd.concat([train, test], axis=0).reset_index(drop=True)

# Apply one hot encoding
combined = pd.get_dummies(combined, columns=remaining_cat, drop_first=True)

# Split back into train and test
train = combined[:train_size].reset_index(drop=True)
test = combined[train_size:].reset_index(drop=True)

print("Train shape after encoding:", train.shape)
print("Test shape after encoding:", test.shape)

Remaining categorical columns to encode: 33
Train shape after encoding: (1458, 228)
Test shape after encoding: (1459, 228)


In [35]:
# Apply StandardScaler
scaler = StandardScaler()

# fit_transform on train - learn and apply
X_train_scaled = scaler.fit_transform(train)

# transform only on test - apply same scaling
X_test_scaled = scaler.transform(test)

# Convert back to dataframes
X_train = pd.DataFrame(X_train_scaled, columns=train.columns)
X_test = pd.DataFrame(X_test_scaled, columns=test.columns)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("\nSample scaled values:")
print(X_train.head(2))

X_train shape: (1458, 228)
X_test shape: (1459, 228)

Sample scaled values:
   MSSubClass  LotFrontage   LotArea  OverallQual  OverallCond  YearBuilt  \
0    0.073426    -0.232336 -0.203934     0.658506    -0.517649   1.052959   
1   -0.871868     0.466541 -0.087252    -0.068293     2.177825   0.158428   

   YearRemodAdd  MasVnrArea  ExterQual  ExterCond  ...  SaleType_ConLI  \
0      0.880362    0.523937   1.061109  -0.238285  ...       -0.058661   
1     -0.428115   -0.570739  -0.689001  -0.238285  ...       -0.058661   

   SaleType_ConLw  SaleType_New  SaleType_Oth  SaleType_WD  \
0       -0.058661     -0.299476     -0.045408     0.388265   
1       -0.058661     -0.299476     -0.045408     0.388265   

   SaleCondition_AdjLand  SaleCondition_Alloca  SaleCondition_Family  \
0               -0.05245             -0.091098             -0.117933   
1               -0.05245             -0.091098             -0.117933   

   SaleCondition_Normal  SaleCondition_Partial  
0              0

In [37]:
# Create directories if needed
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# Save processed data
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)

# Save scaler for future use
joblib.dump(scaler, '../models/scaler.pkl')

print("All files saved successfully!")
print("\nProcessed folder contents:")
print(os.listdir('../data/processed'))
print("\nModels folder contents:")
print(os.listdir('../models'))

All files saved successfully!

Processed folder contents:
['test_ids.csv', 'test_cleaned.csv', 'test_featured.csv', 'train_cleaned.csv', 'X_train.csv', 'y_train.csv', 'X_test.csv', 'train_featured.csv']

Models folder contents:
['scaler.pkl']
